In [2]:
import pandas as pd

df = pd.read_parquet("../data/modeling/driver_race_modeling.parquet")

drivers_per_race = df.groupby("race_id")["driver_id"].nunique()
drivers_per_race.describe()


count    329.000000
mean      20.446809
std        2.116514
min       12.000000
25%       20.000000
50%       20.000000
75%       22.000000
max       24.000000
Name: driver_id, dtype: float64

In [3]:
podium_by_year = df.groupby("year")["is_podium"].mean()
podium_by_year


year
2010    0.120283
2011    0.120853
2012    0.121076
2013    0.131980
2014    0.144737
2015    0.137097
2016    0.132420
2017    0.150000
2018    0.150000
2019    0.150000
2020    0.150000
2021    0.150000
2022    0.150000
2023    0.150000
2024    0.143158
2025    0.144958
Name: is_podium, dtype: float64

In [4]:
top_constructors = (
    df.groupby("constructor_id")["is_podium"]
      .mean()
      .sort_values(ascending=False)
      .head(10)
)
top_constructors


constructor_id
mercedes        0.444094
red_bull        0.416404
ferrari         0.311628
mclaren         0.178849
lotus_f1        0.157534
racing_point    0.052632
aston_martin    0.039648
renault         0.029851
alpine          0.026316
williams        0.026275
Name: is_podium, dtype: float64

In [5]:
dnf_by_year = df.groupby("year")["is_dnf"].mean()
dnf_by_year


year
2010    0.250000
2011    0.158768
2012    0.181614
2013    0.147208
2014    0.200000
2015    0.217742
2016    0.171233
2017    0.237500
2018    0.202381
2019    0.142857
2020    0.167647
2021    0.134091
2022    0.168182
2023    0.309091
2024    0.404211
2025    0.313025
Name: is_dnf, dtype: float64

In [6]:
df.groupby("driver_id")["is_podium"].rolling(5).mean()


driver_id      
aitken     4416    NaN
albon      3696    NaN
           3716    NaN
           3736    NaN
           3756    NaN
                  ... 
zhou       6171    0.0
           6190    0.0
           6210    0.0
           6230    0.0
           6250    0.0
Name: is_podium, Length: 6727, dtype: float64

In [14]:
df = df.sort_values(["constructor_id", "race_date"])

df["constructor_podium_rate_last5"]=(
    df.groupby("constructor_id")["is_podium"]
      .shift(1)
      .rolling(5)
      .mean()
)

df[["constructor_id", "race_date", "is_podium", "constructor_podium_rate_last5"]].head(15)

,constructor_id,race_date,is_podium,constructor_podium_rate_last5
3699,alfa,2019-03-17,0,NaN
3710,alfa,2019-03-17,0,NaN
3719,alfa,2019-03-31,0,NaN
3730,alfa,2019-03-31,0,NaN
3739,alfa,2019-04-14,0,NaN
3750,alfa,2019-04-14,0,0.0
3759,alfa,2019-04-28,0,0.0
3770,alfa,2019-04-28,0,0.0
3779,alfa,2019-05-12,0,0.0
3790,alfa,2019-05-12,0,0.0


In [ ]:
# Step 1: aggregate to constructor-race level
team_race = (
    df.groupby(["constructor_id", "race_id", "race_date"], as_index=False)
      .agg(
          team_podium=("is_podium", "max"),  # 1 if any driver podiumed
      )
)

team_race = team_race.sort_values(["constructor_id", "race_date"])

team_race["constructor_podium_rate_last5"] = (
    team_race.groupby("constructor_id")["team_podium"]
             .transform(lambda x: x.shift(1).rolling(5).mean())
)

df[["constructor_id", "race_date", "is_podium", "constructor_podium_rate_last5"]].head(15)


,constructor_id,race_date,is_podium,constructor_podium_rate_last5
0,alfa,2019-03-17,0,NaN
1,alfa,2019-03-17,0,NaN
2,alfa,2019-03-31,0,NaN
3,alfa,2019-03-31,0,NaN
4,alfa,2019-04-14,0,NaN
5,alfa,2019-04-14,0,NaN
6,alfa,2019-04-28,0,NaN
7,alfa,2019-04-28,0,NaN
8,alfa,2019-05-12,0,NaN
9,alfa,2019-05-12,0,NaN


In [20]:
df = df.merge(
    team_race[["constructor_id", "race_id", "constructor_podium_rate_last5"]],
    on=["constructor_id", "race_id"],
    how="left"
)
df[["constructor_podium_rate_last5", "is_podium"]].corr()



,constructor_podium_rate_last5,is_podium
constructor_podium_rate_last5,1.000000,0.565233
is_podium,0.565233,1.000000


In [11]:

df = df.sort_values(["driver_id", "race_date"])

df["driver_podium_rate_last5"] = (
    df.groupby("driver_id")["is_podium"]
      .shift(1)
      .rolling(5)
      .mean()
)

df[["driver_id", "race_date", "is_podium", "driver_podium_rate_last5"]].head(15)

,driver_id,race_date,is_podium,driver_podium_rate_last5
4416,aitken,2020-12-06,0,NaN
3696,albon,2019-03-17,0,NaN
3716,albon,2019-03-31,0,NaN
3736,albon,2019-04-14,0,NaN
3756,albon,2019-04-28,0,NaN
3776,albon,2019-05-12,0,NaN
3796,albon,2019-05-26,0,0.0
3816,albon,2019-06-09,0,0.0
3836,albon,2019-06-23,0,0.0
3856,albon,2019-06-30,0,0.0


In [12]:
df[["driver_podium_rate_last5", "is_podium"]].corr()


,driver_podium_rate_last5,is_podium
driver_podium_rate_last5,1.000000,0.569817
is_podium,0.569817,1.000000
